# Práctica guiada — De formulario a experiencia de inferencia

En S5 ya construisteis un formulario Streamlit que recoge los 11 atributos y llama a un gateway basado en el contrato de S4. En S6 **no se rediseña el contrato ni se vuelve a implementar el formulario**. El objetivo es convertir esa pantalla funcional en una interfaz operable: mantiene el resultado al haber un rerun, hace visible lo que está ocurriendo y enseña la información justa para interpretar e investigar una predicción.

Trabajad en parejas. Completad los `TODO`, ejecutad las comprobaciones y conservad el notebook como evidencia para el taller. La implementación queda para la Clase 2.

## Entrega de la práctica

Al terminar debéis poder enseñar cuatro decisiones concretas:

1. qué permanece en la sesión, qué se cachea y qué no se retiene;
2. la máquina `idle → loading → success/error`;
3. un plano de pantalla con componentes Streamlit concretos;
4. mensajes seguros para confianza baja, lentitud y error.

No se entrega código de inferencia nuevo. S4 sigue siendo la única frontera de validación y predicción.

## 1. Comprobar el punto de partida de S5

Ejecutad la celda. Debe demostrar que partís del formulario y gateway de S5, no de una segunda implementación. Después anotad una limitación que observaríais al cambiar un widget o al enviar una segunda predicción.

In [1]:
from pathlib import Path

repo_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / 'semana5/modules/05-streamlit-basic-model-ui').is_dir()
)
s5_app = repo_root / 'semana5/modules/05-streamlit-basic-model-ui/solutions/02-first-streamlit/app.py'
source = s5_app.read_text(encoding='utf-8')

checks = {
    'reutiliza st.form': 'st.form(' in source,
    'recoge valores antes de inferir': 'collect_values' in source,
    'aún no usa session_state': 'st.session_state' not in source,
}
print('app de S5:', s5_app)
for label, passed in checks.items():
    print(f"{'✓' if passed else '✗'} {label}")
assert all(checks.values())

# TODO: escribid aquí una limitación observable de S5.
limitacion_s5 = 'Al interactuar con la interfaz tras predecir, el resultado anterior se pierde inmediatamente porque no vive en st.session_state, obligando a reejecutar la inferencia.'
print('Limitación observada:', limitacion_s5)

app de S5: c:\Users\aleja\OneDrive\Escritorio\uni\MLOps\icai-muiaap-operacion-de-modelos\semana5\modules\05-streamlit-basic-model-ui\solutions\02-first-streamlit\app.py
✓ reutiliza st.form
✓ recoge valores antes de inferir
✓ aún no usa session_state
Limitación observada: Al interactuar con la interfaz tras predecir, el resultado anterior se pierde inmediatamente porque no vive en st.session_state, obligando a reejecutar la inferencia.


## 2. Decidir dónde vive cada cosa

Un rerun vuelve a ejecutar el script. Una variable local desaparece; `st.session_state` conserva información de **esa sesión**; `st.cache_resource` reutiliza recursos como un gateway o modelo. Un recurso cacheado global puede compartirse entre usuarios, así que debe ser seguro para concurrencia.

Completad la tabla y justificad cada decisión. No guardéis secretos ni las 11 features en la telemetría. Los valores del último envío solo se conservan para permitir un reintento y no se muestran ni se registran.

| Clave o recurso | Dónde vive | ¿Sobrevive a un rerun? | ¿Se limpia al pulsar «Limpiar»? | Motivo |
| --- | --- | --- | --- | --- |
| `last_state` | st.session_state | Sí | Sí | Conserva la fase y el resultado para volver a mostrarlos tras un rerun |
| `last_values` | st.session_state | Sí | Sí | Retiene los valores del último envío únicamente para permitir un reintento |
| `telemetry` | st.session_state | Sí | No | Acumula métricas agregadas de la sesión |
| gateway / bundle | st.cache_resource | Sí | No | Es un recurso reutilizable compartido entre sesiones |
| respuesta de una petición | st.session_state | Sí | Sí | Pertenece a la sesión y se conserva para mostrar el resultado tras un rerun |

Referencias: [`st.session_state`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state) y [`st.cache_resource`](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.cache_resource).

## 3. Diseñar la pantalla, no solo el estado

La interfaz debe tener cuatro zonas, en este orden:

1. **acción:** el formulario heredado de S5;
2. **operación:** qué está ocurriendo mientras se ejecuta;
3. **resultado:** categoría, confianza y latencia;
4. **trazabilidad:** versiones y `request_id`, visibles solo si se necesitan.

Elegid componentes que tengan una función. `st.columns` organiza, `st.status` comunica una operación, `st.metric` destaca un dato y `st.expander` aplaza el detalle técnico. No uséis una barra de progreso para la confianza: representa avance de una tarea, no certeza del modelo.

In [7]:
# Sustituid cada TODO por el componente y la justificación que usaríais.
# Las claves describen zonas de la interfaz; no son nuevos contratos de ML.

ui_plan = {
    'formulario': {'component': 'st.form', 'reason': 'Agrupa los campos y el envio para evitar inferencias parciales'},
    'estado_operacion': {'component': 'st.status', 'reason': 'Comunica  el estado de la petición'},
    'resumen_resultado': {'component': 'st.metric', 'reason': 'Presenta el resultado de la inferencia, con calidad y confianza.'},
    'mensaje_interpretacion': {'component': 'st.warning', 'reason': 'Advierte al usuario'},
    'trazabilidad': {'component': 'st.expander', 'reason': 'Permite al usuario ver la trazabilidad de la inferencia si lo desea, sin saturar la interfaz.'},
    'separacion_telemetria': {'component': 'st.divider', 'reason': 'Permite separar visualmente la telemetría de la inferencia del resto de la interfaz, para que el usuario pueda distinguirlas.'},
}

allowed_components = {
    'st.form', 'st.status', 'st.empty', 'st.columns', 'st.container',
    'st.metric', 'st.success', 'st.info', 'st.warning', 'st.error',
    'st.expander', 'st.divider', 'st.caption',
}
required_zones = {
    'formulario', 'estado_operacion', 'resumen_resultado',
    'mensaje_interpretacion', 'trazabilidad', 'separacion_telemetria',
}

assert set(ui_plan) == required_zones
for zone, decision in ui_plan.items():
    assert set(decision) == {'component', 'reason'}, zone

pending = [zone for zone, decision in ui_plan.items() if 'TODO' in decision.values()]
invalid = [
    f"{zone}: {decision['component']}"
    for zone, decision in ui_plan.items()
    if 'TODO' not in decision['component'] and decision['component'] not in allowed_components
]
print('Zonas pendientes:', pending or 'ninguna')
print('Componentes no permitidos:', invalid or 'ninguno')
print('Comprobación final: pending == [] and invalid == []')

Zonas pendientes: ninguna
Componentes no permitidos: ninguno
Comprobación final: pending == [] and invalid == []


### Plano mínimo esperado

```text
Título y contexto breve

[ formulario de S5 en dos columnas ]

[ estado de la operación ]

[ Categoría ]  [ Confianza ]  [ Latencia ]
mensaje interpretativo de confianza y latencia
▶ Trazabilidad: versiones y request_id

────────────────────────────────────
Telemetría agregada de la sesión
```

Podéis entregar este plano en el notebook o redibujarlo en draw.io. Justificad cualquier cambio respecto a esta estructura.

## 4. Máquina de estados y recuperación

Completad la tabla. Una respuesta válida con latencia superior a 300 ms sigue en `success`: añade un aviso técnico, no se convierte en error. La confianza es una señal para revisar, no un diagnóstico ni una garantía.

| Estado | Evento de entrada | Zona de pantalla que cambia | Qué se muestra | Acción permitida | Evidencia |
| --- | --- | --- | --- | --- |
| `idle` | Inicio / limpiar | formularios y resultados | mensaje informativo | rellenar formulario y ejecutar | formulario activo sin métricas ni errores en pantalla |
| `loading` | Enviar formulario | parte que muestra que está cargando | contenedor que muestra que está cargando | esperar, nada más | componente st.status visible |
| `success` | Payload válido | resultado | resultados, categoría, confianza, latencia y trazabilidad  | enviar nueva inferencia | resultados y trazabilidad se muestran al seleccionar el preview correspondiente |
| `error` | Excepción controlada | resultado | mensaje seguro de error con st.error | mensaje informativo, corregir los datos y reintentar | alerta visible y el id de la inferencia, y sin trazabilidad |

Incluid una recuperación para entrada inválida, bundle no disponible y timeout. Indicad también qué no debe ver la persona: traceback, rutas internas, secretos o el payload completo.



Entrada inválida: por ejemplo un valor fuera de rango. Se muestra un mensaje seguro de error y se permite corregir los datos y reintentar. No se muestra el traceback, rutas internar, secretos y payload.

Bundle no disponible: Por ejemplo ruta inexistente. se muestra un mensaje con el motivo y se permite reajustar. No se muestra el traceback, rutas internas, secretos y payload.

Timeout: por ejemplo, cuando la inferencia tarda demasiado. Se muestra un mensaje seguro de error y se permite reintentar. No se muestra el traceback, rutas internas, secretos y payload.





## 5. Escribir el copy de una interfaz responsable

Cada mensaje debe decir qué ha ocurrido, qué significa y qué puede hacer la persona. No debéis prometer que una predicción es correcta. Completad textos de una frase, sin terminología interna.

In [2]:
ux_copy = {
    'low_confidence': 'Respuesta válida, pero con confianza baja. Use el resultado con precaución.',
    'slow_success': 'Respuesta válida, pero la inferencia ha tardado más de lo esperado',
    'invalid_input': 'Entrada inválida. Corrija los datos y reintente.',
    'artifact_unavailable': 'El modelo no está disponible temporalmente. Intente de nuevo más tarde.',
}

assert set(ux_copy) == {
    'low_confidence', 'slow_success', 'invalid_input', 'artifact_unavailable'
}
for scenario, message in ux_copy.items():
    assert isinstance(message, str) and message.strip(), scenario

pending_copy = [scenario for scenario, message in ux_copy.items() if 'TODO' in message]
print('Mensajes pendientes:', pending_copy or 'ninguno')
print('Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?')

Mensajes pendientes: ninguno
Revisión humana: ¿el texto evita certeza, traceback y culpabilizar a la persona?


## 6. Preparar el taller

Entregad el notebook con los TODO completados. Antes de cerrar, otra pareja debe poder responder estas preguntas leyendo vuestro trabajo:

- ¿Qué se mantiene al pulsar otro widget y qué se elimina con «Limpiar último resultado»?
- ¿Qué función cargará el gateway una sola vez y cuál será su límite?
- ¿Qué componentes dibujará `render_state()` para `loading`, `success` y `error`?
- ¿Qué ve una persona con confianza baja o una respuesta lenta?
- ¿Dónde aparecen el `request_id` y las versiones sin invadir el resultado?

En el taller se implementarán estas decisiones en `session.py`, `policies.py`, `presentation.py`, `controller.py` y `app.py`.

## Debrief

- ¿Qué limita a S5 aunque el formulario ya funcione?
Que al no gestionar estado con st.session_state, cualquier interacción con otro elemento de la página recarga el script (rerun) y borra el resultado de la predicción anterior.
- ¿Por qué el gateway es un recurso cacheable y el resultado de una petición pertenece a una sesión?
Porque el gateway no cambia y puede ser compartido entre sesiones, mientras que el resultado de una petición es específico de la sesión del usuario y no debe compartirse.
- ¿Qué componente hace visible cada transición sin añadir una segunda lógica de inferencia?
st.status, que comunica el estado de la operación (idle, loading, success, error) sin necesidad de duplicar la lógica de inferencia.
- ¿Por qué categoría, confianza y latencia no tienen el mismo significado?
Categoría es la predicción del modelo, confianza es una medida de certeza de esa predicción y latencia es el tiempo que tarda en completarse la inferencia.
- ¿Qué parte de este diseño se conserva al sustituir el gateway local por HTTP en S7?
La estructura de estados y la lógica de manejo de errores se conservan. Lo único que cambiará es la implementación interna del gateway, que hará llamadas a una API)